# Phase 8 — video anotasi untuk review

Notebook **CPU, render saja** dari hasil final Kaggle Phase 8 Version 2, run `20261015T083731_664817Z`. Tidak menjalankan ulang YOLO, BoT-SORT, ReID, counting, atau resolusi identitas.

**Input Kaggle:** tambahkan notebook output `chrisbiran/phase-8-tgdi-full-video-research` serta dataset `chrisbiran/tgdi-traffic-videos`. Output sumber harus mempunyai manifest SHA `79e8cf19747c63b34f8389d8f525925fd0c69e966f3d12ffb89716a5c89bd0dc`. Pilih accelerator **None/CPU**; model dan GPU tidak diperlukan. Source Python dan delapan geometri berasal dari GitHub atau bootstrap notebook yang terikat hash.

Default Run All merender delapan video. Ubah `VIDEO_IDS` menjadi `['weekday_pagi']` untuk satu video; hasil akan dinyatakan subset, bukan penyelesaian delapan video. Video sumber maupun hasil mesin lama tidak ditimpa.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path(globals().get('REPO_ROOT', Path.cwd())).resolve()
if not (REPO_ROOT / 'notebooks').is_dir() and (REPO_ROOT.parent / 'notebooks').is_dir():
    REPO_ROOT = REPO_ROOT.parent
INPUT_ROOT = globals().get('INPUT_ROOT', '/kaggle/input')
SOURCE_RUN_ROOT = globals().get('SOURCE_RUN_ROOT', None)  # None: temukan output kernel via SHA manifest
SOURCE_VIDEO_ROOT = globals().get('SOURCE_VIDEO_ROOT', None)  # None: cari nama aktual, lalu verifikasi SHA
OUTPUT_ROOT = globals().get('OUTPUT_ROOT', '/kaggle/working/phase_8_annotated_review_v1')
VIDEO_IDS = globals().get('VIDEO_IDS', None)  # None = seluruh delapan; contoh ['weekday_pagi']
PREVIEW_FRAMES = globals().get('PREVIEW_FRAMES', ())  # sampel origin/crossing/akhir juga dibuat otomatis

sys.path.insert(0, str(REPO_ROOT / 'notebooks'))
print({'repo_root': str(REPO_ROOT), 'output_root': str(OUTPUT_ROOT), 'video_ids': VIDEO_IDS or 'ALL_EIGHT'})


## Pemeriksaan software

Tes ini memakai fixture sintetis untuk memeriksa logika tampilan dan penolakan input berubah. Tes bukan eksperimen kualitas model atau ground truth kendaraan. Tidak ada instalasi dependency model/GPU.


In [ ]:
import platform
import cv2
import numpy as np
import unittest
import phase_8_annotation_tests
import phase_8_annotation_batch_tests

print({'python': platform.python_version(), 'opencv': cv2.__version__, 'numpy': np.__version__,
       'execution': 'CPU_RENDER_ONLY_NO_MODEL_OR_COUNTING_RERUN'})
suite = unittest.TestSuite(unittest.defaultTestLoader.loadTestsFromModule(module)
    for module in (phase_8_annotation_tests, phase_8_annotation_batch_tests))
software_result = unittest.TextTestRunner(verbosity=1).run(suite)
if not software_result.wasSuccessful():
    raise RuntimeError('Software guard gagal; anotasi tidak dijalankan')


## Verifikasi bukti dan render

Controller memverifikasi manifest asli, hasil per-video, ledger accepted, audit queue pending, cache pengukuran, relasi komponen, dan geometri berdasarkan SHA. Hanya satu snapshot **machine** asli dipilih per video. Semua frame dibaca dan MP4 hasil didekode ulang. Kegagalan sebagian menyimpan laporan lalu membuat notebook gagal; hasil gagal tidak disebut selesai.


In [ ]:
from phase_8_annotation_batch import run_annotation_batch

ANNOTATION_REPORT = run_annotation_batch(
    repo_root=REPO_ROOT, source_run_root=SOURCE_RUN_ROOT, source_video_root=SOURCE_VIDEO_ROOT,
    input_root=INPUT_ROOT, output_root=OUTPUT_ROOT, video_ids=VIDEO_IDS,
    preview_frames=PREVIEW_FRAMES, progress=print,
)
print({key: ANNOTATION_REPORT[key] for key in (
    'status', 'output_directory', 'selected_video_ids', 'selected_accepted_replayed',
    'selected_pending_retained', 'all_eight_annotation_videos_completed')})


## Cara membaca hasil

Buka tab **Output**, lalu `phase_8_annotated_review_v1/<timestamp-baru>/`. Setiap `<video_id>/` berisi `annotated_review.mp4`, `annotation_report.json`, `passage_display_index.json`, dan PNG preview. Laporan keseluruhan adalah `annotation_batch_report.json`; `input_hashes.json` serta `annotation_retention_manifest.json` mengikat bytes sumber/hasil.

- Pivot memisahkan **kategori × arah**, hanya accepted passage unik. A→B mendekati kamera; B→A menjauhi kamera. Pending merupakan kandidat review, bukan jumlah kendaraan unik, dan tidak masuk subtotal.
- Tampilan merupakan **replay hasil final offline**, ditempatkan pada frame konfirmasi geometri tersimpan. Seluruh 58 event baseline menggunakan bukti kategori setelah crossing/konfirmasi. Panel ini tidak mengklaim hasil sudah tersedia secara online pada frame tersebut.
- Warna bbox/label mengikuti kelas pengukuran mentah saat itu. Kategori final/unit tampil terpisah, sehingga class flip tetap dapat diamati. Frame indeks dimulai dari 0; waktu frame/FPS hanya estimasi, bukan PTS sumber terverifikasi.
- Connector komponen hanya tampil pada frame bukti relasi lokal accepted yang benar-benar tersimpan. Komponen person tidak menambah hitungan motorcycle. Jumlah episode/ID komponen bukan jumlah rider atau penumpang. Kotak kendaraan dan komponen tetap terpisah; BC kendaraan sumber tidak diganti.
- Pergantian ID setelah occlusion belum diperbaiki oleh perubahan anotasi ini. Renderer tidak menciptakan penyatuan identitas baru.
- `0` berarti tidak ada accepted pada kategori yang didukung, bukan bukti kendaraan kategori itu tidak hadir. Pedestrian dan bus/angkot belum lengkap didukung/pemetaan; tampil sebagai N/A.

Baseline seluruh delapan tetap **58 accepted / 1.825 pending**, tanpa review manual baru atau klaim akurasi. Catat feedback dengan `video_id`, frame, ID, alias event/unit, masalah, dan bukti visual.
